<a href="https://colab.research.google.com/github/ViNguyen3/Research-UAV/blob/main/Vi's_n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
                                                         ### 1. Mount Google Drive ###
from google.colab import drive
drive.mount('/content/drive')
### 2. Prepare data ###

# !scp '/content/drive/MyDrive/YOLOV11/data4.zip' '/content/data4.zip'
!cp "/content/drive/MyDrive/Research/YOLO/Data/data4.zip" /content/data4.zip


!unzip '/content/data4.zip' -d '/content/drive/MyDrive/Research/YOLO'

### 3. Install packages ###
ROOT_DIR = '/content/drive/MyDrive/Research/YOLO/data4'

Mounted at /content/drive
Archive:  /content/data4.zip
replace /content/drive/MyDrive/Research/YOLO/data4/.DS_Store? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: /content/drive/MyDrive/Research/YOLO/data4/.DS_Store  
replace /content/drive/MyDrive/Research/YOLO/__MACOSX/data4/._.DS_Store? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: /content/drive/MyDrive/Research/YOLO/__MACOSX/data4/._.DS_Store  
  inflating: /content/drive/MyDrive/Research/YOLO/__MACOSX/data4/._images  
  inflating: /content/drive/MyDrive/Research/YOLO/__MACOSX/data4/._labels  
  inflating: /content/drive/MyDrive/Research/YOLO/data4/images/.DS_Store  
  inflating: /content/drive/MyDrive/Research/YOLO/__MACOSX/data4/images/._.DS_Store  
  inflating: /content/drive/MyDrive/Research/YOLO/__MACOSX/data4/images/._test  
  inflating: /content/drive/MyDrive/Research/YOLO/__MACOSX/data4/images/._train  
  inflating: /content/drive/MyDrive/Research/YOLO/__MACOSX/data4/images/._val  
  inflating: /content/drive

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 30.8 MB/s eta 0:00:00


In [ ]:
import ultralytics
import torch
import os

print("Ultralytics version:", ultralytics.__version__)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics version: 8.4.37
Torch version: 2.10.0+cpu
CUDA available: False


In [ ]:
ROOT_DIR = "/content/drive/MyDrive/Research/VI_YOLO"
DATA_YAML = os.path.join(ROOT_DIR, "config.yaml")

MODEL_NAME = "yolov10n.pt"   # start with n first, easier/faster
EPOCHS = 100
IMGSZ = 960
BATCH = 8
PROJECT = os.path.join(ROOT_DIR, "runs")
NAME = "baseline_yolov10n"

In [ ]:
#Base training
from ultralytics import YOLO

model = YOLO(MODEL_NAME)

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=PROJECT,
    name=NAME,
    pretrained=True
)

Ultralytics 8.4.35 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Research/VI_YOLO/config.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov10n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=baseline_yolov10n, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_m

In [ ]:
#Locating best weight
BEST_PT = os.path.join(PROJECT, NAME, "weights", "best.pt")
print("Best model:", BEST_PT)
print("Exists:", os.path.exists(BEST_PT))

Best model: /content/drive/MyDrive/Research/VI_YOLO/runs/baseline_yolov10n/weights/best.pt
Exists: True


#Stage 2 Distillation

In [ ]:
%%writefile /content/distill_trainer.py
import torch
import torch.nn.functional as F

from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import RANK
from ultralytics.utils.loss import E2ELoss, v8DetectionLoss


class DistillDetectionLoss(v8DetectionLoss):
    def __init__(self, model, teacher_model, lambda_kd=0.5, temperature=2.0, tal_topk=10, tal_topk2=None):
        super().__init__(model, tal_topk=tal_topk, tal_topk2=tal_topk2)
        self.teacher_model = teacher_model.eval()
        self.lambda_kd = lambda_kd
        self.temperature = temperature

        for p in self.teacher_model.parameters():
            p.requires_grad = False

    def _extract_main_pred(self, preds):
        # version-dependent helper
        if isinstance(preds, (list, tuple)):
            return preds[0]
        return preds

    def __call__(self, preds, batch):
        base_loss, base_items = super().__call__(preds, batch)

        with torch.no_grad():
            teacher_preds = self.teacher_model(batch["img"])

        student_main = self._extract_main_pred(preds)
        teacher_main = self._extract_main_pred(teacher_preds)

        # Flatten for KL if needed
        student_logits = student_main.reshape(student_main.shape[0], -1)
        teacher_logits = teacher_main.reshape(teacher_main.shape[0], -1)

        T = self.temperature
        kd_loss = F.kl_div(
            F.log_softmax(student_logits / T, dim=1),
            F.softmax(teacher_logits / T, dim=1),
            reduction="batchmean",
        ) * (T * T)

        total_loss = base_loss + self.lambda_kd * kd_loss
        extra = kd_loss.detach().unsqueeze(0)
        return total_loss, torch.cat([base_items, extra], dim=0)


class DistillE2ELoss(E2ELoss):
    def __init__(self, model, teacher_model, lambda_kd=0.5, temperature=2.0):
        def loss_fn(model, tal_topk=10, tal_topk2=None):
            return DistillDetectionLoss(
                model,
                teacher_model=teacher_model,
                lambda_kd=lambda_kd,
                temperature=temperature,
                tal_topk=tal_topk,
                tal_topk2=tal_topk2,
            )
        super().__init__(model, loss_fn=loss_fn)


class DistillDetectionModel(DetectionModel):
    def __init__(self, cfg, teacher_model, nc, verbose=True):
        super().__init__(cfg, nc=nc, verbose=verbose)
        self._teacher_model = teacher_model

    def init_criterion(self):
        return DistillE2ELoss(
            self,
            teacher_model=self._teacher_model,
            lambda_kd=0.5,
            temperature=2.0,
        )


class DistillTrainer(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        # teacher = bigger model
        teacher = YOLO("yolov10l.pt").model
        model = DistillDetectionModel(
            cfg,
            teacher_model=teacher,
            nc=self.data["nc"],
            verbose=verbose and RANK == -1,
        )
        if weights:
            model.load(weights)
        return model

Writing /content/distill_trainer.py


In [ ]:
## DISTILLATION TRAINING
from ultralytics import YOLO
from distill_trainer import DistillTrainer

student = YOLO("yolov10n.pt")

student.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=IMGSZ,
    batch=BATCH,
    project=PROJECT,
    name="distill_yolov10n_from_yolov10l",
    trainer=DistillTrainer,
)

Ultralytics 8.4.35 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Research/VI_YOLO/config.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov10n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=distill_yolov10n_from_yolov10l, nbs=64, nms=False, opset=None, optimize=False, optimizer=aut

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e2c54452a20>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
        

#Spatial Attention

In [ ]:
!pip install -U ultralytics

In [ ]:
import ultralytics
print("Ultralytics version:", ultralytics.__version__)

Ultralytics version: 8.4.37


In [ ]:
%%writefile /content/attention_block.py
import torch
import torch.nn as nn

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        padding = kernel_size // 2
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_map = torch.mean(x, dim=1, keepdim=True)
        max_map, _ = torch.max(x, dim=1, keepdim=True)
        attn = torch.cat([avg_map, max_map], dim=1)
        attn = self.sigmoid(self.conv(attn))
        return x * attn

Writing /content/attention_block.py


In [ ]:
%%writefile /content/attention_trainer.py
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import RANK
from attention_block import SpatialAttention
import types


class AttentionDetectionModel(DetectionModel):
    def __init__(self, cfg, nc, verbose=True):
        super().__init__(cfg, nc=nc, verbose=verbose)

        target_idx = 4

        if len(self.model) > target_idx:
            block = self.model[target_idx]
            block.sa = SpatialAttention()

            original_forward = block.forward

            def forward_with_attention(self_block, x):
                x = original_forward(x)
                x = self_block.sa(x)
                return x

            # store as named attribute too
            block.forward_with_attention = types.MethodType(forward_with_attention, block)
            block.forward = block.forward_with_attention


class AttentionTrainer(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = AttentionDetectionModel(
            cfg,
            nc=self.data["nc"],
            verbose=verbose and RANK == -1,
        )
        if weights is not None:
            model.load(weights)
        return model

    def save_model(self):
        # Skip checkpoint serialization because monkeypatched forwards
        # do not deserialize cleanly in standard Ultralytics reload flow.
        return

    def final_eval(self):
        # Skip final validation on best/last checkpoints
        return

Overwriting /content/attention_trainer.py


In [ ]:
import sys, importlib

if "/content" not in sys.path:
    sys.path.append("/content")

import attention_block
import attention_trainer

importlib.reload(attention_block)
importlib.reload(attention_trainer)

from attention_trainer import AttentionTrainer
from ultralytics import YOLO

In [ ]:
# model = YOLO("yolov10n.pt")
# model.train(
#     data=DATA_YAML,
#     epochs=1,
#     imgsz=640,
#     batch=2,
#     project=PROJECT,
#     name="attention_yolov10n_debug",
#     trainer=AttentionTrainer,
#     device="cpu"
# )

from ultralytics import YOLO

model = YOLO("yolov10n.pt")

try:
    results = model.train(
        data=DATA_YAML,
        epochs=1,
        imgsz=640,
        batch=2,
        project=PROJECT,
        name="attention_yolov10n_debug_skip_final",
        trainer=AttentionTrainer,
        device="cpu"
    )
except FileNotFoundError as e:
    if "weights/last.pt" in str(e):
        print("Training finished. Ignoring post-training checkpoint cleanup error for monkeypatched attention model.")
        results = None
    else:
        raise

Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Research/VI_YOLO/config.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov10n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=attention_yolov10n_debug_skip_final2, nbs=64, nms=False, opset=None, optimize=Fal

In [ ]:
m = YOLO("yolov10n.pt")
custom = AttentionTrainer(overrides={"model": "yolov10n.pt", "data": DATA_YAML})
net = custom.get_model(cfg=m.model.yaml, weights=None)

print(type(net.model[4]).__name__)
print(hasattr(net.model[4], "sa"))

Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Research/VI_YOLO/config.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov10n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_

#Vine Specific Box Term

In [ ]:
%%writefile /content/vine_loss_trainer.py
import torch

from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import RANK
from ultralytics.utils.loss import E2ELoss, v8DetectionLoss


class VineDetectionLoss(v8DetectionLoss):
    def __init__(self, model, lambda_vine=0.1, tal_topk=10, tal_topk2=None):
        super().__init__(model, tal_topk=tal_topk, tal_topk2=tal_topk2)
        self.lambda_vine = lambda_vine

    def _extract_main_pred(self, preds):
        if isinstance(preds, (list, tuple)):
            return preds[0]
        return preds

    def __call__(self, preds, batch):
        base_loss, base_items = super().__call__(preds, batch)

        pred_tensor = self._extract_main_pred(preds)

        # placeholder interpretation; may need adjustment for your version
        flat = pred_tensor.reshape(pred_tensor.shape[0], -1)
        vine_penalty = (flat.std(dim=1).mean()) * 0.0  # safe stub to keep it runnable

        total_loss = base_loss + self.lambda_vine * vine_penalty
        extra = vine_penalty.detach().unsqueeze(0)
        return total_loss, torch.cat([base_items, extra], dim=0)


class VineE2ELoss(E2ELoss):
    def __init__(self, model, lambda_vine=0.1):
        def loss_fn(model, tal_topk=10, tal_topk2=None):
            return VineDetectionLoss(model, lambda_vine=lambda_vine, tal_topk=tal_topk, tal_topk2=tal_topk2)
        super().__init__(model, loss_fn=loss_fn)


class VineDetectionModel(DetectionModel):
    def init_criterion(self):
        return VineE2ELoss(self, lambda_vine=0.1)


class VineTrainer(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = VineDetectionModel(cfg, nc=self.data["nc"], verbose=verbose and RANK == -1)
        if weights:
            model.load(weights)
        return model

Writing /content/vine_loss_trainer.py


In [ ]:
from ultralytics import YOLO
from vine_loss_trainer import VineTrainer

model = YOLO("yolov10n.pt")
model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=IMGSZ,
    batch=BATCH,
    project=PROJECT,
    name="vine_loss_yolov10n",
    trainer=VineTrainer,
)

Ultralytics 8.4.36 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Research/VI_YOLO/config.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov10n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=vine_loss_yolov10n, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_m

KeyboardInterrupt: 

#Evaluation

In [ ]:
# =========================
# YOLO detection evaluation
# Precision, Recall, F1, mAP50, mAP50-95, per-class summary
# =========================

!pip install -q pyyaml pandas

from ultralytics import YOLO
import pandas as pd
import math

MODEL_PATH = "yolov10n.pt"   # change to your trained model path if needed
DATA_YAML = DATA_YAML        # uses your existing variable

model = YOLO(MODEL_PATH)

# Run official Ultralytics validation
metrics = model.val(
    data=DATA_YAML,
    imgsz=640,
    batch=2,
    device="cpu",
    split="val",
    plots=False
)

# Built-in metrics from Ultralytics
precision = float(metrics.box.mp)      # mean precision
recall = float(metrics.box.mr)         # mean recall
map50 = float(metrics.box.map50)       # mAP@0.50
map5095 = float(metrics.box.map)       # mAP@0.50:0.95

# Custom F1 from precision/recall
f1 = 0.0 if (precision + recall) == 0 else 2 * precision * recall / (precision + recall)

print("=== Overall Metrics ===")
print(f"Precision : {precision:.6f}")
print(f"Recall    : {recall:.6f}")
print(f"F1 Score  : {f1:.6f}")
print(f"mAP50     : {map50:.6f}")
print(f"mAP50-95  : {map5095:.6f}")

# Per-class summary
summary = metrics.summary()
summary_df = pd.DataFrame(summary)

print("\n=== Per-class summary ===")
display(summary_df)

# Optional save
summary_df.to_csv("per_class_metrics.csv", index=False)

overall_df = pd.DataFrame([{
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "map50": map50,
    "map50_95": map5095
}])
overall_df.to_csv("overall_metrics.csv", index=False)

##Image level FPR block

In [ ]:
# ==========================================
# Image-level TP / TN / FP / FN and FPR code
# ==========================================

import yaml
from pathlib import Path

CONF_THRES = 0.25   # adjust if needed

def load_yaml(path):
    with open(path, "r") as f:
        return yaml.safe_load(f)

def resolve_split_path(data_yaml, split="val"):
    cfg = load_yaml(data_yaml)
    split_path = cfg[split]

    # handle relative paths
    yaml_dir = Path(data_yaml).resolve().parent
    split_path = Path(split_path)
    if not split_path.is_absolute():
        split_path = (yaml_dir / split_path).resolve()

    return split_path

def get_label_path_from_image(img_path: Path):
    # standard YOLO layout:
    # images/val/abc.jpg -> labels/val/abc.txt
    parts = list(img_path.parts)
    try:
        i = parts.index("images")
        parts[i] = "labels"
    except ValueError:
        raise ValueError(f"Could not infer label path from image path: {img_path}")
    label_path = Path(*parts).with_suffix(".txt")
    return label_path

def image_has_gt_object(label_path: Path):
    if not label_path.exists():
        return False
    text = label_path.read_text().strip()
    return len(text) > 0

def image_has_prediction(result, conf_thres=0.25):
    if result.boxes is None or len(result.boxes) == 0:
        return False
    confs = result.boxes.conf.cpu().numpy()
    return (confs >= conf_thres).any()

val_dir = resolve_split_path(DATA_YAML, split="val")
image_paths = sorted([
    p for p in val_dir.rglob("*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
])

print(f"Found {len(image_paths)} validation images")

results = model.predict(
    source=[str(p) for p in image_paths],
    conf=CONF_THRES,
    device="cpu",
    verbose=False
)

TP = TN = FP = FN = 0

rows = []

for img_path, result in zip(image_paths, results):
    label_path = get_label_path_from_image(img_path)
    gt_positive = image_has_gt_object(label_path)
    pred_positive = image_has_prediction(result, CONF_THRES)

    if gt_positive and pred_positive:
        TP += 1
        outcome = "TP"
    elif gt_positive and not pred_positive:
        FN += 1
        outcome = "FN"
    elif not gt_positive and pred_positive:
        FP += 1
        outcome = "FP"
    else:
        TN += 1
        outcome = "TN"

    rows.append({
        "image": str(img_path),
        "label_file": str(label_path),
        "gt_positive": gt_positive,
        "pred_positive": pred_positive,
        "outcome": outcome
    })

image_eval_df = pd.DataFrame(rows)

precision_img = TP / (TP + FP) if (TP + FP) > 0 else 0.0
recall_img = TP / (TP + FN) if (TP + FN) > 0 else 0.0
f1_img = 2 * precision_img * recall_img / (precision_img + recall_img) if (precision_img + recall_img) > 0 else 0.0
fpr_img = FP / (FP + TN) if (FP + TN) > 0 else 0.0

print("\n=== Image-level Confusion Counts ===")
print(f"TP: {TP}")
print(f"TN: {TN}")
print(f"FP: {FP}")
print(f"FN: {FN}")

print("\n=== Image-level Metrics ===")
print(f"Precision: {precision_img:.6f}")
print(f"Recall   : {recall_img:.6f}")
print(f"F1 Score : {f1_img:.6f}")
print(f"FPR      : {fpr_img:.6f}")

display(image_eval_df)
image_eval_df.to_csv("image_level_eval.csv", index=False)

In [ ]:
print("Precision = TP / (TP + FP)")
print("Recall    = TP / (TP + FN)")
print("F1        = 2PR / (P + R)")
print("FPR       = FP / (FP + TN)   # image-level in this implementation")

##Helper function for EVALUATION

In [ ]:
def evaluate_model(model_path, data_yaml, imgsz=640, batch=2, conf_thres=0.25, device="cpu"):
    from ultralytics import YOLO
    import pandas as pd
    import yaml
    from pathlib import Path

    model = YOLO(model_path)

    metrics = model.val(
        data=data_yaml,
        imgsz=imgsz,
        batch=batch,
        device=device,
        split="val",
        plots=False
    )

    precision = float(metrics.box.mp)
    recall = float(metrics.box.mr)
    map50 = float(metrics.box.map50)
    map5095 = float(metrics.box.map)
    f1 = 0.0 if (precision + recall) == 0 else 2 * precision * recall / (precision + recall)

    def load_yaml(path):
        with open(path, "r") as f:
            return yaml.safe_load(f)

    def resolve_split_path(data_yaml, split="val"):
        cfg = load_yaml(data_yaml)
        split_path = cfg[split]
        yaml_dir = Path(data_yaml).resolve().parent
        split_path = Path(split_path)
        if not split_path.is_absolute():
            split_path = (yaml_dir / split_path).resolve()
        return split_path

    def get_label_path_from_image(img_path: Path):
        parts = list(img_path.parts)
        i = parts.index("images")
        parts[i] = "labels"
        return Path(*parts).with_suffix(".txt")

    def image_has_gt_object(label_path: Path):
        if not label_path.exists():
            return False
        return len(label_path.read_text().strip()) > 0

    def image_has_prediction(result, conf_thres=0.25):
        if result.boxes is None or len(result.boxes) == 0:
            return False
        confs = result.boxes.conf.cpu().numpy()
        return (confs >= conf_thres).any()

    val_dir = resolve_split_path(data_yaml, split="val")
    image_paths = sorted([
        p for p in val_dir.rglob("*")
        if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
    ])

    results = model.predict(
        source=[str(p) for p in image_paths],
        conf=conf_thres,
        device=device,
        verbose=False
    )

    TP = TN = FP = FN = 0
    rows = []

    for img_path, result in zip(image_paths, results):
        label_path = get_label_path_from_image(img_path)
        gt_positive = image_has_gt_object(label_path)
        pred_positive = image_has_prediction(result, conf_thres)

        if gt_positive and pred_positive:
            TP += 1
            outcome = "TP"
        elif gt_positive and not pred_positive:
            FN += 1
            outcome = "FN"
        elif not gt_positive and pred_positive:
            FP += 1
            outcome = "FP"
        else:
            TN += 1
            outcome = "TN"

        rows.append({
            "image": str(img_path),
            "gt_positive": gt_positive,
            "pred_positive": pred_positive,
            "outcome": outcome
        })

    precision_img = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall_img = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1_img = 2 * precision_img * recall_img / (precision_img + recall_img) if (precision_img + recall_img) > 0 else 0.0
    fpr_img = FP / (FP + TN) if (FP + TN) > 0 else 0.0

    return {
        "overall_detection_metrics": {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "map50": map50,
            "map50_95": map5095,
        },
        "image_level_metrics": {
            "TP": TP,
            "TN": TN,
            "FP": FP,
            "FN": FN,
            "precision": precision_img,
            "recall": recall_img,
            "f1": f1_img,
            "fpr": fpr_img,
        },
        "per_class_summary_df": pd.DataFrame(metrics.summary()),
        "image_level_df": pd.DataFrame(rows),
    }

In [ ]:
out = evaluate_model(
    model_path="yolov10n.pt",
    data_yaml=DATA_YAML,
    imgsz=640,
    batch=2,
    conf_thres=0.25,
    device="cpu"
)

print(out["overall_detection_metrics"])
print(out["image_level_metrics"])
display(out["per_class_summary_df"])
display(out["image_level_df"])